In [37]:
"""
四方对比：Projection / Local Repair / Retrain / IFRU
=====================================================
加载各方已保存的 embedding，统一计算以下指标：
  - PFR_multi
  - Target Score Drop
  - Non-target Score Change
  - Selectivity Ratio
  - Overlap@K vs Retrain
  - Overlap@K vs Original（遗忘效果显著性）
=====================================================
"""

import os
import numpy as np
import pandas as pd
import torch

# ─────────────────────────────────────────────────────────────
# 路径
# ─────────────────────────────────────────────────────────────
BASE           = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR"
DATA_PATH      = "/Users/yubinhao/ml-1m/Amazon Review/数据集/Amazon_Reviews_Processed"

EMBED_PATH     = f"{BASE}/模型文件/embeddings_best.pt"
MAPPING_PATH   = f"{BASE}/模型文件/id_mappings.pt"
BASIS_PATH     = f"{BASE}/pattern_basis/pattern_basis.pt"
ITEM_SETS_PATH = f"{BASE}/pattern_basis/pattern_item_sets.pt"
INFLUENCE_PATH = f"{BASE}/pattern_influence/influence_scores.pt"
UNLEARN_PATH   = f"{BASE}/遗忘实验/embeddings_unlearned.pt"
REPAIR_PATH    = f"{BASE}/local_repair/embeddings_repaired.pt"
RETRAIN_PATH   = f"{BASE}/Retrain/embeddings_retrain.pt"
IFRU_PATH      = f"{BASE}/IFRU/ifru_results.pt"
SAVE_DIR       = f"{BASE}/四方对比"
os.makedirs(SAVE_DIR, exist_ok=True)

TOP_K            = 8
MIN_INTERACTIONS = 5
EPS              = 1e-10
MIN_NT           = 0.01   # Selectivity Ratio 分母最小值


# ─────────────────────────────────────────────────────────────
# 1. 加载所有 embedding
# ─────────────────────────────────────────────────────────────
print("[1] 加载 embeddings ...")

# Original
orig_ckpt         = torch.load(EMBED_PATH,  map_location="cpu", weights_only=False)
user_emb_orig     = orig_ckpt["user_embeddings"].numpy()   # (n_users, d)
item_emb_orig     = orig_ckpt["item_embeddings"].numpy()   # (n_items, d)

# Projection (Unlearn)
unlearn_ckpt      = torch.load(UNLEARN_PATH, map_location="cpu", weights_only=False)
user_emb_proj     = unlearn_ckpt["user_emb_after"]         # (n_users, d)

# Local Repair
repair_ckpt       = torch.load(REPAIR_PATH,  map_location="cpu", weights_only=False)
user_emb_repair   = repair_ckpt["user_emb_repaired"]       # (n_users, d)
item_emb_repair   = repair_ckpt["item_emb_repaired"]       # (n_items, d)

# Retrain
retrain_ckpt      = torch.load(RETRAIN_PATH, map_location="cpu", weights_only=False)
user_emb_retrain  = retrain_ckpt["user_embeddings"].numpy()
item_emb_retrain  = retrain_ckpt["item_embeddings"].numpy()

# IFRU
ifru_ckpt         = torch.load(IFRU_PATH,    map_location="cpu", weights_only=False)
user_emb_ifru     = ifru_ckpt["user_emb_ifru"]             # (n_users, d)
item_emb_ifru     = ifru_ckpt["item_emb_ifru"]             # (n_items, d)

print(f"  Original:     user {user_emb_orig.shape},  item {item_emb_orig.shape}")
print(f"  Projection:   user {user_emb_proj.shape}")
print(f"  Local Repair: user {user_emb_repair.shape}, item {item_emb_repair.shape}")
print(f"  Retrain:      user {user_emb_retrain.shape}, item {item_emb_retrain.shape}")
print(f"  IFRU:         user {user_emb_ifru.shape}, item {item_emb_ifru.shape}")


# ─────────────────────────────────────────────────────────────
# 2. 加载 pattern basis、influence、B_u
# ─────────────────────────────────────────────────────────────
print("\n[2] 加载辅助数据 ...")

mapping           = torch.load(MAPPING_PATH,   map_location="cpu", weights_only=False)
user2idx          = mapping["user2idx"]

basis_ckpt        = torch.load(BASIS_PATH,     map_location="cpu", weights_only=False)
pattern_basis     = basis_ckpt["pattern_basis"].numpy()

sets_ckpt         = torch.load(ITEM_SETS_PATH, map_location="cpu", weights_only=False)
pattern_item_sets = sets_ckpt["pattern_item_sets"]

infl_ckpt         = torch.load(INFLUENCE_PATH, map_location="cpu", weights_only=False)
influence_matrix  = infl_ckpt["influence_matrix"]
sampled_uids      = infl_ckpt["sampled_uids"]
hq_patterns       = infl_ckpt["hq_patterns"]

# B_u
df = pd.read_csv(DATA_PATH, usecols=["user_id", "item_id", "rating"])
uc = df.groupby("user_id").size()
df = df[df["user_id"].isin(uc[uc >= MIN_INTERACTIONS].index)].copy()
df["uid"] = df["user_id"].map(user2idx)
df["iid"] = df["item_id"].map(
    {it: i for i, it in enumerate(sorted(df["item_id"].unique()))}
)
# 用 mapping 里的 item2idx 保证一致
df["iid"] = df["item_id"].map(mapping["item2idx"])
df.dropna(subset=["uid", "iid"], inplace=True)
df["uid"] = df["uid"].astype(int)
df["iid"] = df["iid"].astype(int)
Bu = df.groupby("uid")["iid"].apply(set).to_dict()
print(f"  B_u 构造完成，用户数: {len(Bu)}")


# ─────────────────────────────────────────────────────────────
# 3. 计算 S_u、I_target、I_non_target
# ─────────────────────────────────────────────────────────────
print("\n[3] 计算 S_u、I_target(u)、I_non_target(u) ...")

mu_u    = influence_matrix.mean(axis=1)
sigma_u = influence_matrix.std(axis=1)
tau_u   = mu_u + sigma_u

Su_dict           = {}
I_target_dict     = {}
I_non_target_dict = {}

for idx, uid in enumerate(sampled_uids):
    scores    = influence_matrix[idx]
    threshold = tau_u[idx]
    sorted_ki = np.argsort(scores)[::-1]
    targets   = [ki for ki in sorted_ki if scores[ki] > threshold]
    if not targets:
        continue
    Su_dict[uid] = targets
    bu           = Bu.get(uid, set())
    union_Ik     = set()
    for ki in targets:
        union_Ik.update(pattern_item_sets[hq_patterns[ki]])
    I_tgt  = bu & union_Ik
    I_ntgt = bu - I_tgt
    if I_tgt:
        I_target_dict[uid]     = list(I_tgt)
        I_non_target_dict[uid] = list(I_ntgt)

valid_uids = [uid for uid in sampled_uids
              if uid in I_target_dict and len(I_target_dict[uid]) > 0]
print(f"  有效用户数: {len(valid_uids)}")


# ─────────────────────────────────────────────────────────────
# 4. 定义各方打分函数
# ─────────────────────────────────────────────────────────────
# 注意：Projection 用 item_emb_orig（只改了 user emb）
#       Local Repair 用 item_emb_repair（同时改了 user + item emb）
#       Retrain / IFRU 各自用自己的 item emb

def score(eu, xi):
    return eu @ xi   # 内积，LightGCN 无偏置

methods = {
    "Projection":    (user_emb_proj,    item_emb_orig),
    "Local Repair":  (user_emb_repair,  item_emb_repair),
    "Retrain":       (user_emb_retrain, item_emb_retrain),
    "IFRU":          (user_emb_ifru,    item_emb_ifru),
}


# ─────────────────────────────────────────────────────────────
# 5. 逐用户计算指标
# ─────────────────────────────────────────────────────────────
print("\n[4] 计算指标 ...")

records = {m: {
    "pfr":        [],
    "tsd":        [],   # Target Score Drop
    "ntsc":       [],   # Non-target Score Change
    "sel":        [],   # Selectivity Ratio
    "overlap_orig":    [],   # Overlap@K vs Original
    "overlap_retrain": [],   # Overlap@K vs Retrain
} for m in methods}

for uid in valid_uids:
    eu_orig   = user_emb_orig[uid]
    I_tgt     = I_target_dict[uid]
    I_ntgt    = I_non_target_dict.get(uid, [])
    ki_list   = Su_dict[uid]

    # Original Top-K（用于 Overlap vs Orig）
    topk_orig = set(np.argpartition(item_emb_orig @ eu_orig, -TOP_K)[-TOP_K:])

    # Retrain Top-K（用于 Overlap vs Retrain，固定参考）
    eu_ret    = user_emb_retrain[uid]
    topk_ret  = set(np.argpartition(item_emb_retrain @ eu_ret, -TOP_K)[-TOP_K:])

    # Original scores on target / non-target items
    s_orig_tgt  = item_emb_orig[I_tgt]  @ eu_orig
    s_orig_ntgt = item_emb_orig[I_ntgt] @ eu_orig if I_ntgt else np.array([])

    for m_name, (ue, ie) in methods.items():
        eu_m = ue[uid]

        # ── PFR_multi ────────────────────────────────────────
        A_before = sum(abs(eu_orig @ pattern_basis[hq_patterns[ki]]) for ki in ki_list)
        A_after  = sum(abs(eu_m   @ pattern_basis[hq_patterns[ki]]) for ki in ki_list)
        pfr = 1.0 - A_after / (A_before + EPS) if A_before > 0 else 0.0
        records[m_name]["pfr"].append(pfr)

        # ── Target Score Drop ─────────────────────────────────
        s_m_tgt = ie[I_tgt] @ eu_m
        tsd = float((s_orig_tgt - s_m_tgt).mean())
        records[m_name]["tsd"].append(tsd)

        # ── Non-target Score Change ───────────────────────────
        if len(I_ntgt) > 0:
            s_m_ntgt = ie[I_ntgt] @ eu_m
            ntsc = float(np.abs(s_orig_ntgt - s_m_ntgt).mean())
        else:
            ntsc = 0.0
        records[m_name]["ntsc"].append(ntsc)

        # ── Selectivity Ratio ─────────────────────────────────
        if ntsc > MIN_NT:
            records[m_name]["sel"].append(tsd / ntsc)
        else:
            records[m_name]["sel"].append(np.nan)

        # ── Overlap@K vs Original ─────────────────────────────
        topk_m = set(np.argpartition(ie @ eu_m, -TOP_K)[-TOP_K:])
        records[m_name]["overlap_orig"].append(len(topk_orig & topk_m) / TOP_K)

        # ── Overlap@K vs Retrain ──────────────────────────────
        records[m_name]["overlap_retrain"].append(len(topk_ret & topk_m) / TOP_K)

print(f"  计算完成，用户数: {len(valid_uids)}")


# ─────────────────────────────────────────────────────────────
# 6. 汇总打印
# ─────────────────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"  四方对比汇总（n={len(valid_uids)} 用户）")
print(f"{'='*80}")

metrics_def = [
    ("PFR_multi",               "pfr",            "↑越高越好（遗忘彻底度）"),
    ("Target Score Drop",       "tsd",            "↑越高越好（目标偏好被压低）"),
    ("Non-target Score Change", "ntsc",           "↓越低越好（非目标偏好保护）"),
    ("Selectivity Ratio",       "sel",            "↑越高越好（选择性）"),
    ("Overlap@K vs Original",   "overlap_orig",   "↓越低越好（遗忘显著性）"),
    ("Overlap@K vs Retrain",    "overlap_retrain","↑越高越好（接近retrain程度）"),
]

col_w = 14
header = f"  {'指标':<30}" + "".join(f"{m:>{col_w}}" for m in methods)
print(header)
print(f"  {'-'*78}")

for metric_name, key, note in metrics_def:
    row = f"  {metric_name:<30}"
    for m_name in methods:
        vals = [v for v in records[m_name][key] if not np.isnan(v)]
        mean = np.mean(vals) if vals else float("nan")
        row += f"{mean:>{col_w}.4f}"
    print(row + f"   {note}")

print(f"{'='*80}")


# # ─────────────────────────────────────────────────────────────
# # 7. 保存为 CSV
# # ─────────────────────────────────────────────────────────────
# print(f"\n[5] 保存结果至 {SAVE_DIR} ...")

# rows = []
# for metric_name, key, note in metrics_def:
#     row = {"指标": metric_name, "说明": note}
#     for m_name in methods:
#         vals = [v for v in records[m_name][key] if not np.isnan(v)]
#         row[m_name + "_mean"] = round(np.mean(vals), 4) if vals else float("nan")
#         row[m_name + "_std"]  = round(np.std(vals),  4) if vals else float("nan")
#     rows.append(row)

# df_result = pd.DataFrame(rows)
# df_result.to_csv(os.path.join(SAVE_DIR, "四方对比结果.csv"), index=False, encoding="utf-8-sig")

# # 同时保存完整 per-user 数据
# torch.save({"records": records, "valid_uids": valid_uids},
#            os.path.join(SAVE_DIR, "四方对比_per_user.pt"))

# print("  四方对比结果.csv     — 汇总均值/标准差")
# print("  四方对比_per_user.pt — 逐用户原始数据")
# print("\n[完成]")

[1] 加载 embeddings ...
  Original:     user (27794, 64),  item (144853, 64)
  Projection:   user (27794, 64)
  Local Repair: user (27794, 64), item (144853, 64)
  Retrain:      user (27794, 64), item (144853, 64)
  IFRU:         user (27794, 64), item (144853, 64)

[2] 加载辅助数据 ...
  B_u 构造完成，用户数: 27794

[3] 计算 S_u、I_target(u)、I_non_target(u) ...
  有效用户数: 889

[4] 计算指标 ...
  计算完成，用户数: 889

  四方对比汇总（n=889 用户）
  指标                                Projection  Local Repair       Retrain          IFRU
  ------------------------------------------------------------------------------
  PFR_multi                             0.9767        0.9894        0.5195        0.0055   ↑越高越好（遗忘彻底度）
  Target Score Drop                     2.3735        4.2775        7.7701        0.7766   ↑越高越好（目标偏好被压低）
  Non-target Score Change               0.8682        0.0161        0.7327        2.7571   ↓越低越好（非目标偏好保护）
  Selectivity Ratio                     3.1997      246.1767       15.5504        0.9112   ↑越高越好（选择性）
  O

In [33]:
retrain_ckpt = torch.load(RETRAIN_PATH, map_location="cpu", weights_only=False)
print(retrain_ckpt.keys())

dict_keys(['user_embeddings', 'item_embeddings', 'loss'])
